# BLM5135 — Track 1 training runbook

Run cell-by-cell or use **Runtime → Run all**. Outputs go directly to your Drive at `/content/drive/MyDrive/dl_project_outputs/`, so if the session dies you can come back and resume any run.

**Before starting:** confirm the runtime is A100 via *Runtime → Change runtime type → A100 GPU*. If only T4/V100/L4 is available, training will work but be slower than the budget assumes.

## 1. Mount Drive and confirm dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
DATASET_ROOT = '/content/drive/MyDrive/dataset'
JSON_PATH = f'{DATASET_ROOT}/dataset.json'
assert os.path.isfile(JSON_PATH), f'dataset.json not found at {JSON_PATH}'
with open(JSON_PATH) as fh:
    nimg = len(json.load(fh)['images'])
print(f'Drive mounted. dataset.json has {nimg} samples.')

## 2. Confirm GPU is A100 (or at least a CUDA device)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Clone the repo (or update if already cloned) and set up output dirs

In [ ]:
import os, subprocess
REPO_URL = 'https://github.com/amirkiarafiei/deep-learning-project.git'
REPO_DIR = '/content/deep-learning-project'
DRIVE_OUT = '/content/drive/MyDrive/dl_project_outputs'
HF_CACHE = f'{DRIVE_OUT}/hf_cache'

if os.path.isdir(f'{REPO_DIR}/.git'):
    subprocess.run(['git', 'fetch', 'origin'], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'reset', '--hard', 'origin/main'], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--depth=1', REPO_URL, REPO_DIR], check=True)

for d in [
    DRIVE_OUT, HF_CACHE,
    f'{DRIVE_OUT}/results/track1/object',
    f'{DRIVE_OUT}/results/track1/event',
    f'{DRIVE_OUT}/results/track1/attribute',
    f'{DRIVE_OUT}/results/track1/multitask',
    f'{DRIVE_OUT}/results/track1/object_smoke',
]:
    os.makedirs(d, exist_ok=True)

subprocess.run(['git', 'log', '-1', '--oneline'], cwd=REPO_DIR)
print(f'Repo at {REPO_DIR}, outputs at {DRIVE_OUT}, HF cache at {HF_CACHE}.')

## 4. Install Python deps

Colab usually has a CUDA torch preinstalled — we only need to add `timm` and a few small libs.

In [ ]:
!pip install -q 'timm==1.0.27' 'scikit-learn>=1.4,<2.0' 'PyYAML>=6.0,<7.0' 'tqdm>=4.66'
import torch, timm, sklearn
print('torch  ', torch.__version__, '| cuda:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')
print('timm   ', timm.__version__)
print('sklearn', sklearn.__version__)

## 5. Smoke test — 200 train / 100 val, 2 epochs, batch 32

Should finish in well under 5 minutes on A100. Confirms the full pipeline (data loader, model forward, autocast bf16, loss, optimizer, checkpoint write to Drive) works end-to-end before committing to the 4 real runs.

In [ ]:
import os, subprocess
env = os.environ.copy()
env['HF_HOME'] = HF_CACHE
env['PYTHONUNBUFFERED'] = '1'  # stream subprocess stdout to the cell live
cmd = [
    'python', '-u', 'train.py',
    '--config', 'configs/track1_object.yaml',
    '--output-dir', f'{DRIVE_OUT}/results/track1/object_smoke',
    '--dataset-root', DATASET_ROOT,
    '--json-path', JSON_PATH,
    '--run-name', 'object_smoke',
    '--subset-train', '200',
    '--subset-val', '100',
    '--epochs', '2',
    '--batch-size', '32',
    '--num-workers', '2',
]
subprocess.run(cmd, cwd=REPO_DIR, env=env, check=True)
print('\n✅ Smoke test complete.')

## 6. Full Track-1 training — four configs in sequence

Each: ConvNeXt-Tiny + 30 epochs + early stop on val avg macro-F1 + patience 10. Outputs to Drive every epoch so any session loss bounds the cost to one epoch (resume with `--resume <output_dir>/checkpoints/last.pt`).

**Run order:** object → event → attribute → multitask. Each is independent.

In [ ]:
import os, subprocess, time
env = os.environ.copy()
env['HF_HOME'] = HF_CACHE
env['PYTHONUNBUFFERED'] = '1'  # stream subprocess stdout to the cell live

RUNS = ['object', 'event', 'attribute', 'multitask']
RESULTS_BASE = f'{DRIVE_OUT}/results/track1_v2'   # v2 outputs separate from v1
for fam in RUNS:
    print(f'\n=== Training v2: {fam} ===', flush=True)
    t0 = time.time()
    out_dir = f'{RESULTS_BASE}/{fam}'
    cmd = [
        'python', '-u', 'train.py',
        '--config', f'configs/track1_{fam}.yaml',
        '--output-dir', out_dir,
        '--dataset-root', DATASET_ROOT,
        '--json-path', JSON_PATH,
        '--num-workers', '2',
    ]
    last_pt = f'{out_dir}/checkpoints/last.pt'
    if os.path.isfile(last_pt):
        print(f'   (resuming from {last_pt})', flush=True)
        cmd += ['--resume', last_pt]
    subprocess.run(cmd, cwd=REPO_DIR, env=env, check=True)
    print(f'=== {fam} done in {(time.time() - t0)/60:.1f} min ===', flush=True)

print('\n✅ All four v2 trainings complete.')

## 7. Evaluation on the test split + qualitative figures

Writes per-class CSVs, summary JSON, full prediction matrix, and ≥20 qualitative example PNGs + a combined PDF per run.

In [ ]:
import os, subprocess
env = os.environ.copy()
env['HF_HOME'] = HF_CACHE
env['PYTHONUNBUFFERED'] = '1'

RESULTS_BASE = f'{DRIVE_OUT}/results/track1_v2'

def run(cmd, label):
    print(f'  → {label}', flush=True)
    subprocess.run(cmd, cwd=REPO_DIR, env=env, check=True)

for fam in ['object', 'event', 'attribute', 'multitask']:
    out_dir = f'{RESULTS_BASE}/{fam}'
    ckpt = f'{out_dir}/checkpoints/best.pt'
    if not os.path.isfile(ckpt):
        print(f'(skip {fam}: no best.pt)', flush=True)
        continue
    print(f'\n=== Phase 2 eval pipeline: {fam} ===', flush=True)

    common = ['--config', f'configs/track1_{fam}.yaml',
              '--ckpt', ckpt,
              '--output-dir', out_dir,
              '--dataset-root', DATASET_ROOT,
              '--json-path', JSON_PATH]

    # Step 1: eval on val → dumps predictions_val.pt + eval_val.json + per_class_*_val.csv
    run(['python', '-u', 'eval.py'] + common + ['--split', 'val'],
        '1/6 eval val (flat 0.5)')

    # Step 2: tune per-class thresholds on val
    pred_val = f'{out_dir}/metrics/predictions_val.pt'
    thr_json = f'{out_dir}/metrics/thresholds_val.json'
    run(['python', '-u', '-m', 'src.scripts.tune_thresholds',
         '--predictions', pred_val, '--output', thr_json],
        '2/6 per-class threshold sweep on val')

    # Step 3: eval on test, flat 0.5 (baseline numbers; for direct v1↔v2 comparison)
    run(['python', '-u', 'eval.py'] + common + ['--split', 'test'],
        '3/6 eval test (flat 0.5)')

    # Step 4: eval on test with val-tuned per-class thresholds (final reportable numbers)
    run(['python', '-u', 'eval.py'] + common + ['--split', 'test', '--thresholds', thr_json],
        '4/6 eval test (val-tuned thresholds)')

    # Step 5: qualitative figures at flat 0.5 (matches v1 figures for direct comparison)
    run(['python', '-u', 'visualize.py'] + common + ['--split', 'test', '--num-samples', '20'],
        '5/6 visualize 20 samples (flat 0.5 — qualitative/)')

    # Step 6: qualitative figures using val-tuned thresholds (matches tuned metrics)
    run(['python', '-u', 'visualize.py'] + common + ['--split', 'test', '--num-samples', '20',
                                                       '--thresholds', thr_json],
        '6/6 visualize 20 samples (val-tuned — qualitative_tuned/)')

print('\n✅ Phase 2 eval + visualize complete.')
print(f'\nResults under {RESULTS_BASE}/<fam>/')
print('  metrics/eval_val.json         — val flat baseline')
print('  metrics/eval_test.json        — test flat baseline (compare to v1)')
print('  metrics/eval_test_tuned.json  — test with val-tuned per-class thresholds (final)')
print('  metrics/thresholds_val.json   — per-class thresholds picked on val')
print('  metrics/per_class_<fam>_{val,test,test_tuned}.csv')
print('  qualitative/                  — figures at flat 0.5')
print('  qualitative_tuned/            — figures using val-tuned thresholds')

## 8. Pin requirements for submission

Captures the actual library versions used in this Colab session into `requirements.txt`. This is what the course rubric requires.

Copy the output into the local repo's `requirements.txt` and commit before zipping the submission.

In [ ]:
!pip freeze | grep -iE '^(torch|torchvision|timm|scikit-learn|PyYAML|tqdm|numpy|matplotlib|Pillow|requests)=='

# Track 2 — v3 (A3 Hybrid): Cleanup + LDAM-DRW + cRT + LA + TTA

**This section is fully self-contained.** On a fresh runtime, run the cells from here **in order** without touching anything above. No need to run the Track-1 cells.

Pipeline:
1. **Setup** — Drive mount, GPU check, clone repo, install deps, load Gemini key.
2. **Predict v1 multi-task on train** — re-uses your existing v1.0.0 checkpoint on Drive.
3. **Rank noisy labels** — cleanlab-style strong-disagreement ranking.
4. **Gemini relabel** — top-500 most suspect train samples (val relabel optional & skipped by default).
5. **Reconcile** — conservative AND-rule → `dataset_v3_clean.json`.
6. **Train** — LDAM-DRW + cRT, ~3 h A100.
7. **Eval** — flat / +LA / +LA+TTA+thresholds (thresholds tuned on ORIGINAL val for honest test transfer).

**Prerequisites:**
- Colab A100 runtime selected (*Runtime → Change runtime type → A100*).
- `GEMINI_API_KEY` added to Colab Secrets (left sidebar 🔑 icon). Get key at https://aistudio.google.com/apikey.
- v1 multi-task checkpoint exists at `/content/drive/MyDrive/dl_project_outputs/results/track1_v1/multitask/checkpoints/best.pt` (it should from earlier Track-1 runs).

## v3.1 — Setup (mount Drive, clone repo, install deps, load API key)

Single cell that does everything. Idempotent — safe to re-run if you reconnect.

In [ ]:
# === Track 2 v3 setup — fully self-contained ===
import os, subprocess, sys

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Confirm GPU
print('\n--- nvidia-smi ---')
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'])
import torch
print(f'\ntorch.cuda.is_available(): {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Paths
REPO_URL    = 'https://github.com/amirkiarafiei/deep-learning-project.git'
REPO_DIR    = '/content/deep-learning-project'
DRIVE_OUT   = '/content/drive/MyDrive/dl_project_outputs'
HF_CACHE    = f'{DRIVE_OUT}/hf_cache'
DATASET_ROOT= '/content/drive/MyDrive/dataset'
JSON_PATH   = f'{DATASET_ROOT}/dataset.json'

# Track 2's v3 (A3 Hybrid) — outputs live under results/track2_v3/ on both
# repo and Drive. (Renamed 2026-05-27 from results/track1_v3/ which was a
# naming bug — Track 2 work was being stored under the Track 1 prefix.)
V3_BASE     = f'{DRIVE_OUT}/results/track2_v3'
CLEANUP_DIR = f'{V3_BASE}/cleanup'
V3_OUT      = f'{V3_BASE}/multitask'
V1_CKPT     = f'{DRIVE_OUT}/results/track1_v1/multitask/checkpoints/best.pt'

# Sanity-check critical paths
assert os.path.isfile(JSON_PATH), f'dataset.json not found at {JSON_PATH}'
assert os.path.isfile(V1_CKPT), (
    f'v1 multi-task checkpoint not found at {V1_CKPT}. '
    f'You need to have run Track-1 first (or restored the v1.0.0 artifacts to Drive).'
)
print(f'Dataset OK at {JSON_PATH}')
print(f'v1 multi-task ckpt OK at {V1_CKPT}')

# If you have OLD Drive artifacts at results/track1_v3/ from before the rename,
# rename them once so future cells find them:
LEGACY_V3 = f'{DRIVE_OUT}/results/track1_v3'
if os.path.isdir(LEGACY_V3) and not os.path.isdir(V3_BASE):
    print(f'\nRenaming legacy {LEGACY_V3} -> {V3_BASE} (one-time migration)')
    os.rename(LEGACY_V3, V3_BASE)

# Clone or update repo
if os.path.isdir(f'{REPO_DIR}/.git'):
    subprocess.run(['git', 'fetch', 'origin'], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'reset', '--hard', 'origin/main'], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--depth=1', REPO_URL, REPO_DIR], check=True)
print('\nCurrent commit:', end=' ')
subprocess.run(['git', 'log', '-1', '--oneline'], cwd=REPO_DIR)

# Output directories on Drive
for d in [DRIVE_OUT, HF_CACHE, CLEANUP_DIR, V3_OUT,
          f'{V3_OUT}/checkpoints', f'{V3_OUT}/logs',
          f'{V3_OUT}/metrics', f'{V3_OUT}/plots',
          f'{V3_OUT}/qualitative']:
    os.makedirs(d, exist_ok=True)

# Install all deps (Track 1 + v3 extras). Idempotent — pip will no-op if already installed.
print('\n--- pip install (Track 1 + v3 extras) ---')
deps = [
    'timm==1.0.27', 'scikit-learn>=1.4,<2.0', 'PyYAML>=6.0,<7.0', 'tqdm>=4.66',
    'cleanlab>=2.6', 'google-genai>=1.0', 'python-dotenv>=1.0', 'pydantic>=2.0',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + deps, check=True)

import timm, sklearn
print(f'torch    {torch.__version__}  cuda={torch.cuda.is_available()}')
print(f'timm     {timm.__version__}')
print(f'sklearn  {sklearn.__version__}')

# Load Gemini API key from Colab Secrets (or fall back to .env)
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    print('\nGEMINI_API_KEY loaded from Colab Secrets.')
except Exception as e:
    env_path = f'{REPO_DIR}/.env'
    if os.path.isfile(env_path):
        for line in open(env_path):
            if line.startswith('GEMINI_API_KEY='):
                os.environ['GEMINI_API_KEY'] = line.split('=', 1)[1].strip()
                print(f'GEMINI_API_KEY loaded from {env_path}.')
                break
    else:
        raise RuntimeError(
            f'GEMINI_API_KEY not found. Either add it via Colab Secrets (🔑 icon) or '
            f'create {env_path} with: GEMINI_API_KEY=<your-key>.  ({e})'
        )

# Subprocess env preset — reused by every following cell.
ENV = os.environ.copy()
ENV['HF_HOME'] = HF_CACHE
ENV['PYTHONUNBUFFERED'] = '1'

def run(cmd, label=None):
    if label: print(f'\n→ {label}', flush=True)
    subprocess.run(cmd, cwd=REPO_DIR, env=ENV, check=True)

print('\n✅ v3.1 setup complete.')

## v3.2 — Predict v1 multi-task on TRAIN

Re-use v1's `best.pt` to dump logits + targets on the training split into `predictions_train.pt`. ~5 min on A100.

In [ ]:
run([
    'python', '-u', '-m', 'src.scripts.predict_for_cleanlab',
    '--config', 'configs/track1_multitask.yaml',
    '--ckpt', V1_CKPT,
    '--output-dir', CLEANUP_DIR,
    '--dataset-root', DATASET_ROOT,
    '--json-path', JSON_PATH,
    '--batch-size', '128',
    '--num-workers', '2',
], label='predict v1 multi-task on train')

## v3.3 — Rank training samples by suspected label noise

Cleanlab-style ranking. Top-500 suspect samples go to Gemini in the next step. ~30s CPU.

In [ ]:
run([
    'python', '-u', '-m', 'src.scripts.find_noisy_labels',
    '--predictions', f'{CLEANUP_DIR}/predictions_train.pt',
    '--output', f'{CLEANUP_DIR}/noisy_candidates.json',
    '--top-k', '500',
], label='rank by noise')

## v3.4 — Gemini relabel of top-500 suspect train samples

~10 min wall clock, ~$0.30 API. Append-only JSONL; `--resume` makes this idempotent if the cell is re-run after an interruption.

In [ ]:
run([
    'python', '-u', '-m', 'src.scripts.gemini_relabel',
    '--dataset-json', JSON_PATH,
    '--dataset-root', DATASET_ROOT,
    '--candidates', f'{CLEANUP_DIR}/noisy_candidates.json',
    '--output', f'{CLEANUP_DIR}/gemini_relabels_train.jsonl',
    '--max-samples', '500',
    '--resume',
], label='gemini relabel top-500 train candidates')

## v3.5 — (OPTIONAL) Relabel validation samples

**Default: SKIPPED.** Reason: thresholds are tuned on the ORIGINAL val set (step v3.7) so they transfer cleanly to the uncleaned test set. Relabeling val only affects training-time early-stopping; gains are marginal and risk a val↔test distribution mismatch.

Set `SKIP_VAL_RELABEL = False` below if you want to enable this as an ablation (~$1.50 extra API, ~20 min).

In [ ]:
SKIP_VAL_RELABEL = True

if not SKIP_VAL_RELABEL:
    run([
        'python', '-u', '-m', 'src.scripts.gemini_relabel',
        '--dataset-json', JSON_PATH,
        '--dataset-root', DATASET_ROOT,
        '--split', 'val',
        '--output', f'{CLEANUP_DIR}/gemini_relabels_val.jsonl',
        '--resume',
    ], label='gemini relabel entire val set')
else:
    print('Skipping val relabel — thresholds will be tuned on original val in step v3.7.')

## v3.6 — Reconcile relabels into a cleaned dataset

Conservative AND-rule: a label is flipped only when (cleanlab high-confidence disagreement) AND (Gemini reports `confidence="high"`) AND (Gemini's labels differ from the original). Test split is never touched. Writes `dataset_v3_clean.json` + a summary of how many labels changed.

In [ ]:
import os
val_relabels = f'{CLEANUP_DIR}/gemini_relabels_val.jsonl'
cmd = [
    'python', '-u', '-m', 'src.scripts.reconcile_labels',
    '--dataset-json', JSON_PATH,
    '--train-relabels', f'{CLEANUP_DIR}/gemini_relabels_train.jsonl',
    '--output',         f'{CLEANUP_DIR}/dataset_v3_clean.json',
    '--summary',        f'{CLEANUP_DIR}/reconcile_summary.json',
]
if os.path.isfile(val_relabels):
    cmd += ['--val-relabels', val_relabels]
run(cmd, label='reconcile relabels → dataset_v3_clean.json')

# Quick peek at the result
import json
summary = json.load(open(f'{CLEANUP_DIR}/reconcile_summary.json'))
print()
print(json.dumps(summary, indent=2, ensure_ascii=False))

## v3.7 — Train multi-task v3 on the cleaned dataset

LDAM-DRW (loss-side class balance, DRW kicks in at epoch 5) → cRT (5 epochs of head-only retraining with class-aware sampling on plain BCE, after Phase A early-stops). ~3 hours A100.

Resume support: cell is safe to re-run after a disconnect. If `last.pt` exists for v3, it auto-resumes from the next epoch.

In [ ]:
import time
t0 = time.time()
cmd = [
    'python', '-u', 'train.py',
    '--config', 'configs/track2_v3_multitask.yaml',
    '--output-dir', V3_OUT,
    '--dataset-root', DATASET_ROOT,
    '--json-path', f'{CLEANUP_DIR}/dataset_v3_clean.json',
    '--num-workers', '2',
]
last_pt = f'{V3_OUT}/checkpoints/last.pt'
if os.path.isfile(last_pt):
    print(f'   (resuming from {last_pt})', flush=True)
    cmd += ['--resume', last_pt]
run(cmd, label='train v3 (LDAM-DRW Phase A + cRT)')
print(f'\n✅ v3 training done in {(time.time() - t0)/60:.1f} min.')

## v3.8 — Evaluate v3 in three flavours

Three test-set evaluations, written to separate JSON files for direct comparison in the IEEE report:

1. **`eval_test.json`** — flat 0.5 threshold, no LA, no TTA. Apples-to-apples vs v1.
2. **`eval_test_LA.json`** — + multi-label logit adjustment.
3. **`eval_test_FINAL.json`** — + LA + 4-way dihedral TTA + per-class thresholds tuned on ORIGINAL val. The headline number we report.

Threshold tuning happens on ORIGINAL val (uncleaned) so the tuned thresholds match the test-set label distribution. Cleaned val is only used during training for early-stopping signal.

In [ ]:
# Pick the better of best.pt (Phase A) and crt_best.pt (cRT phase)
crt_best = f'{V3_OUT}/checkpoints/crt_best.pt'
ckpt = crt_best if os.path.isfile(crt_best) else f'{V3_OUT}/checkpoints/best.pt'
print(f'Using checkpoint: {ckpt}')

# IMPORTANT: we use the ORIGINAL JSON_PATH for evaluation (not the cleaned one)
# so val + test label distributions match the test-time real-world distribution.
COMMON = ['--config', 'configs/track2_v3_multitask.yaml',
          '--ckpt', ckpt,
          '--output-dir', V3_OUT,
          '--dataset-root', DATASET_ROOT,
          '--json-path', JSON_PATH]
log_priors = f'{V3_OUT}/metrics/log_priors.json'

import shutil
METRICS = f'{V3_OUT}/metrics'

def _stash(stage):
    """Move eval_test.json (+ matching per_class_*_test.csv) out of the way
    so the next eval.py invocation doesn't overwrite them. eval.py only adds
    a `_tuned` suffix when --thresholds is passed; without it, every run writes
    to the bare `eval_test.json`. We rename to `_{stage}` between runs.
    """
    src = f'{METRICS}/eval_test.json'
    if os.path.isfile(src):
        shutil.move(src, f'{METRICS}/eval_test_{stage}.json')
    for f in os.listdir(METRICS):
        if f.startswith('per_class_') and f.endswith('_test.csv'):
            shutil.move(f'{METRICS}/{f}', f'{METRICS}/{f[:-4]}_{stage}.csv')

# Step 1: eval val (flat 0.5) — dumps predictions_val.pt for threshold tuning
run(['python', '-u', 'eval.py'] + COMMON + ['--split', 'val'],
    label='1/6 eval val (flat 0.5 — predictions for tuning)')

# Step 2: tune per-class thresholds on ORIGINAL val
run(['python', '-u', '-m', 'src.scripts.tune_thresholds',
     '--predictions', f'{V3_OUT}/metrics/predictions_val.pt',
     '--output',      f'{V3_OUT}/metrics/thresholds_val.json'],
    label='2/6 per-class threshold sweep on original val')
thresholds = f'{V3_OUT}/metrics/thresholds_val.json'

# Step 3: test (flat 0.5) — baseline number for v1↔v3 comparison
run(['python', '-u', 'eval.py'] + COMMON + ['--split', 'test'],
    label='3/6 eval test (flat 0.5)')
_stash('flat')

# Step 4: test + logit adjustment only
run(['python', '-u', 'eval.py'] + COMMON + ['--split', 'test',
     '--logit-adjust', log_priors, '--logit-adjust-tau', '1.0'],
    label='4/6 eval test (+ LA)')
_stash('LA')

# Step 5: test + LA + TTA + val-tuned thresholds (HEADLINE)
run(['python', '-u', 'eval.py'] + COMMON + ['--split', 'test',
     '--logit-adjust', log_priors, '--logit-adjust-tau', '1.0',
     '--tta', '4',
     '--thresholds', thresholds],
    label='5/6 eval test (+ LA + TTA + val-tuned thresholds) — headline')
# This one writes eval_test_tuned.json (suffix from --thresholds) — rename to FINAL.
shutil.move(f'{METRICS}/eval_test_tuned.json',
            f'{METRICS}/eval_test_FINAL.json')

# Step 6: qualitative figures (val-tuned thresholds → matches the headline metric)
run(['python', '-u', 'visualize.py'] + COMMON + ['--split', 'test',
     '--num-samples', '20', '--thresholds', thresholds],
    label='6/6 visualize 20 qualitative samples')

print('\n✅ v3 evaluation complete.')
print(f'\nReport-relevant files (under {V3_OUT}/):')
print('  metrics/eval_test_flat.json    — flat 0.5 (vs v1 apples-to-apples)')
print('  metrics/eval_test_LA.json      — + logit adjustment')
print('  metrics/eval_test_FINAL.json   — + LA + TTA + thresholds (HEADLINE)')
print('  metrics/per_class_<fam>_test_{flat,LA}.csv & _test_tuned.csv')
print('  metrics/thresholds_val.json    — per-class thresholds picked on original val')
print('  metrics/log_priors.json        — class priors used for LA')
print('  qualitative_tuned/             — 20 PNGs + combined PDF')

# Track 2 v4 — A2 isolation: cleanup + v1's exact training recipe

**Goal.** Disentangle whether the 108-flip Gemini cleanup helped or hurt, independent of v3's broken loss-side stack.

**Setup.** Same architecture as v1. Loss = plain BCE with `pos_weight clamp_max=50` (v1's setting, **not** v3's 10). No LDAM, no cRT, no LA, no TTA, no per-class thresholds. Only difference vs v1 = the cleaned dataset (`dataset_v3_clean.json`).

**Cost.** ~3h A100 train + ~5 min eval.

**Decision table after this runs:**
| v4 avg macro-F1 | Conclusion |
|---|---|
| ≥ 0.290 | Cleanup helped → ship v4, update Track-2 conclusion |
| 0.260–0.280 | Cleanup neutral → v1 stays headline |
| < 0.260 | Cleanup hurt (additive-bias hypothesis confirmed) → v1 stays headline, stronger negative-result story |

**Run order (fresh runtime).** This section is fully self-contained — no prior cell required. Just run **v4.1 → v4.2 → v4.3** in order. v4 does need the cleaned dataset (`dataset_v3_clean.json`) to exist on Drive — v4.1 will fail with a clear error if it's missing, pointing you at the v3.6 cell to produce it.

In [ ]:
# === v4.1 — Setup (fully self-contained, no prior cells required) ===
# Idempotent: safe to re-run.
import os, subprocess, sys

# --- Mount Drive ---
from google.colab import drive
drive.mount('/content/drive')

# --- Confirm GPU ---
print('\n--- nvidia-smi ---')
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'])
import torch
print(f'\ntorch.cuda.is_available(): {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# --- Paths ---
REPO_URL     = 'https://github.com/amirkiarafiei/deep-learning-project.git'
REPO_DIR     = '/content/deep-learning-project'
DRIVE_OUT    = '/content/drive/MyDrive/dl_project_outputs'
HF_CACHE     = f'{DRIVE_OUT}/hf_cache'
DATASET_ROOT = '/content/drive/MyDrive/dataset'
JSON_PATH    = f'{DATASET_ROOT}/dataset.json'

# v3 cleanup outputs (where dataset_v3_clean.json lives)
CLEANUP_DIR  = f'{DRIVE_OUT}/results/track2_v3/cleanup'
CLEAN_JSON   = f'{CLEANUP_DIR}/dataset_v3_clean.json'

# v4 output base
V4_OUT       = f'{DRIVE_OUT}/results/track2_v4/multitask'

os.makedirs(HF_CACHE, exist_ok=True)

# --- Sanity-check critical paths ---
assert os.path.isfile(JSON_PATH), f'dataset.json not found at {JSON_PATH}'
assert os.path.isfile(CLEAN_JSON), (
    f'\n*** Cleaned dataset not found at {CLEAN_JSON} ***\n'
    f'v4 needs the Gemini-cleaned dataset produced by v3.6 (Reconcile). '
    f'You need to have run cells v3.1 through v3.6 at least once on this '
    f'Drive account before running v4.\n'
    f'If you have the file at a different path, set CLEAN_JSON manually here.'
)
print(f'\nDataset OK at {JSON_PATH}')
print(f'Cleaned dataset OK at {CLEAN_JSON}')

# --- Clone or update repo ---
if os.path.isdir(f'{REPO_DIR}/.git'):
    subprocess.run(['git', 'fetch', 'origin'], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'reset', '--hard', 'origin/main'], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--depth=1', REPO_URL, REPO_DIR], check=True)
print('\nCurrent commit:', end=' ')
subprocess.run(['git', 'log', '-1', '--oneline'], cwd=REPO_DIR)

# --- Output directories on Drive ---
for d in [V4_OUT, f'{V4_OUT}/checkpoints', f'{V4_OUT}/logs',
          f'{V4_OUT}/metrics', f'{V4_OUT}/plots', f'{V4_OUT}/qualitative']:
    os.makedirs(d, exist_ok=True)

# --- Install Python deps ---
print('\n--- pip install (Track 1 deps; v4 uses v1 recipe so no v3 extras needed) ---')
deps = [
    'timm==1.0.27', 'scikit-learn>=1.4,<2.0', 'PyYAML>=6.0,<7.0', 'tqdm>=4.66',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + deps, check=True)

import timm, sklearn
print(f'\ntorch    {torch.__version__}  cuda={torch.cuda.is_available()}')
print(f'timm     {timm.__version__}')
print(f'sklearn  {sklearn.__version__}')

# --- Subprocess env preset reused by v4.2 and v4.3 ---
ENV = os.environ.copy()
ENV['HF_HOME'] = HF_CACHE
ENV['PYTHONUNBUFFERED'] = '1'

def run(cmd, label=None):
    if label: print(f'\n→ {label}', flush=True)
    subprocess.run(cmd, cwd=REPO_DIR, env=ENV, check=True)

print(f'\n✅ v4 setup complete. Outputs will land under {V4_OUT}/')

In [ ]:
# === v4.2 — Train ConvNeXt-Tiny multitask on the CLEANED dataset with v1's recipe ===
# Uses globals defined in v4.1 setup (V4_OUT, CLEAN_JSON, DATASET_ROOT, JSON_PATH, run, ENV).
# Auto-resumes from last.pt on Drive if a prior runtime got interrupted.
import os, time

t0 = time.time()
cmd = [
    'python', '-u', 'train.py',
    '--config', 'configs/track2_v4_clean_v1recipe_multitask.yaml',
    '--output-dir', V4_OUT,
    '--dataset-root', DATASET_ROOT,
    '--json-path', CLEAN_JSON,
    '--num-workers', '2',
]
last_pt = f'{V4_OUT}/checkpoints/last.pt'
if os.path.isfile(last_pt):
    print(f'   (resuming from {last_pt})', flush=True)
    cmd += ['--resume', last_pt]
run(cmd, label='train v4 (cleanup + v1 recipe)')
print(f'\n✅ v4 training done in {(time.time() - t0)/60:.1f} min.')

In [ ]:
# === v4.3 — Evaluate: val (flat) → tune thresholds → test (flat + val-tuned) ===
# Uses globals defined in v4.1 setup (V4_OUT, DATASET_ROOT, JSON_PATH, run, ENV).
import os, json, shutil

V4_CKPT = f'{V4_OUT}/checkpoints/best.pt'
assert os.path.isfile(V4_CKPT), f'v4 best.pt not found at {V4_CKPT}'

# IMPORTANT: eval on ORIGINAL dataset.json so test labels match v1's eval distribution.
COMMON_V4 = ['--config', 'configs/track2_v4_clean_v1recipe_multitask.yaml',
             '--ckpt', V4_CKPT,
             '--output-dir', V4_OUT,
             '--dataset-root', DATASET_ROOT,
             '--json-path', JSON_PATH]

# 1. eval val (flat) — dumps predictions_val.pt for threshold tuning
run(['python', '-u', 'eval.py'] + COMMON_V4 + ['--split', 'val'],
    label='1/4 v4 eval val (flat 0.5)')

# 2. tune per-class thresholds on val
run(['python', '-u', '-m', 'src.scripts.tune_thresholds',
     '--predictions', f'{V4_OUT}/metrics/predictions_val.pt',
     '--output',      f'{V4_OUT}/metrics/thresholds_val.json'],
    label='2/4 v4 per-class threshold sweep on val')

# 3. eval test (flat 0.5) — primary comparison vs v1
run(['python', '-u', 'eval.py'] + COMMON_V4 + ['--split', 'test'],
    label='3/4 v4 eval test (flat 0.5) — primary')

# 4. eval test with val-tuned thresholds (secondary)
run(['python', '-u', 'eval.py'] + COMMON_V4 + ['--split', 'test',
     '--thresholds', f'{V4_OUT}/metrics/thresholds_val.json'],
    label='4/4 v4 eval test (val-tuned thresholds) — secondary')

# Summary comparison vs v1 and v3
print('\n\n=== v4 vs v1 vs v3 (test-set macro F1) ===')
print(f'{"variant":35s}  {"object":>8s}  {"event":>8s}  {"attr":>8s}  {"AVG":>8s}')
v4_flat = json.load(open(f'{V4_OUT}/metrics/eval_test.json'))
v4_tun  = json.load(open(f'{V4_OUT}/metrics/eval_test_tuned.json'))
for label, s in [('v1 multitask (flat 0.5)',     None),
                 ('v3 flat 0.5',                 None),
                 ('v4 flat 0.5 (cleanup+v1)',    v4_flat),
                 ('v4 val-tuned (cleanup+v1)',   v4_tun)]:
    if s is None:
        if 'v1' in label:
            o, e, a = 0.2850, 0.2861, 0.2404
        else:
            o, e, a = 0.2246, 0.2589, 0.2238
    else:
        o = s['object']['macro_f1']; e = s['event']['macro_f1']; a = s['attribute']['macro_f1']
    print(f'{label:35s}  {o:8.4f}  {e:8.4f}  {a:8.4f}  {(o+e+a)/3:8.4f}')

# Track 3 v5 — DINOv2-Base (frozen) + cross-attention fusion + ML-Decoder

**Hypothesis under test.** v1's ImageNet-1k pretraining is the bottleneck. DINOv2-Base (LVD-142M SSL) should lift macro F1 if representations are the limit.

**Architecture.** Frozen DINOv2-Base extracts 256 patch tokens per image. Cross-attention fusion (A→B and B→A) produces a 513-token joint sequence. Per-family ML-Decoders emit per-class logits via learnable class queries. Only ~5M params train.

**Outputs.**
- Drive: `/content/drive/MyDrive/dl_project_outputs/results/track3_v5/{object,event,attribute,multitask}/`
- Local (after `git pull`): same paths under repo root, modulo the heavy artifacts which the `.gitignore` excludes.

**Robustness.** Each cell:
1. Checks `last.pt` on Drive and auto-resumes if present.
2. Saves to Drive every epoch, so a runtime disconnect at most loses the current in-flight epoch.
3. Skips families whose `best.pt` already exists, so re-running the cell after a partial run is safe.
4. Uses bf16 autocast on A100 to fit DINOv2 in memory.

**Run order (fresh runtime).** This section is fully self-contained — you do **not** need to run any cell from the Track 1 or Track 2 sections first. Just run **v5.1 → v5.2 → v5.3** in order.

In [ ]:
# === v5.1 — Setup (fully self-contained, no prior cells required) ===
# Idempotent: safe to re-run on the same runtime, and safe to run as the
# first cell on a fresh runtime without touching anything above.
import os, subprocess, sys

# --- Mount Drive ---
from google.colab import drive
drive.mount('/content/drive')

# --- Confirm GPU ---
print('\n--- nvidia-smi ---')
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'])
import torch
print(f'\ntorch.cuda.is_available(): {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# --- Paths ---
REPO_URL     = 'https://github.com/amirkiarafiei/deep-learning-project.git'
REPO_DIR     = '/content/deep-learning-project'
DRIVE_OUT    = '/content/drive/MyDrive/dl_project_outputs'
DATASET_ROOT = '/content/drive/MyDrive/dataset'
JSON_PATH    = f'{DATASET_ROOT}/dataset.json'

# Persist torch.hub cache (incl. DINOv2 weights ~340 MB) on Drive so a fresh
# runtime doesn't re-download.
TORCH_HUB_CACHE = f'{DRIVE_OUT}/torch_hub_cache'
HF_CACHE        = f'{DRIVE_OUT}/hf_cache'
os.makedirs(TORCH_HUB_CACHE, exist_ok=True)
os.makedirs(HF_CACHE, exist_ok=True)
os.environ['TORCH_HOME'] = TORCH_HUB_CACHE
os.environ['HF_HOME']    = HF_CACHE

# --- Sanity-check critical paths ---
assert os.path.isfile(JSON_PATH), f'dataset.json not found at {JSON_PATH}'
print(f'\nDataset OK at {JSON_PATH}')

# --- Clone or update repo ---
if os.path.isdir(f'{REPO_DIR}/.git'):
    subprocess.run(['git', 'fetch', 'origin'], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'reset', '--hard', 'origin/main'], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--depth=1', REPO_URL, REPO_DIR], check=True)
print('\nCurrent commit:', end=' ')
subprocess.run(['git', 'log', '-1', '--oneline'], cwd=REPO_DIR)

# --- Output directories on Drive ---
V5_BASE     = f'{DRIVE_OUT}/results/track3_v5'
V5_FAMILIES = ['object', 'event', 'attribute', 'multitask']
for fam in V5_FAMILIES:
    for sub in ('checkpoints', 'logs', 'metrics', 'plots', 'qualitative'):
        os.makedirs(f'{V5_BASE}/{fam}/{sub}', exist_ok=True)

# --- Install Python deps ---
# v5 doesn't need the v3 Gemini / Cleanlab stack, but installing them is
# cheap (pip no-ops if already installed) and lets the user run v3/v4
# cells later in the same runtime if they want to.
print('\n--- pip install (Track 1 + v3 extras + v5 deps) ---')
deps = [
    'timm==1.0.27', 'scikit-learn>=1.4,<2.0', 'PyYAML>=6.0,<7.0', 'tqdm>=4.66',
    'cleanlab>=2.6', 'google-genai>=1.0', 'python-dotenv>=1.0', 'pydantic>=2.0',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + deps, check=True)

import timm, sklearn
print(f'\ntorch    {torch.__version__}  cuda={torch.cuda.is_available()}')
print(f'timm     {timm.__version__}')
print(f'sklearn  {sklearn.__version__}')

# --- Subprocess env preset reused by every following cell ---
ENV = os.environ.copy()
ENV['HF_HOME']    = HF_CACHE
ENV['TORCH_HOME'] = TORCH_HUB_CACHE
ENV['PYTHONUNBUFFERED'] = '1'

def run(cmd, label=None):
    if label: print(f'\n→ {label}', flush=True)
    subprocess.run(cmd, cwd=REPO_DIR, env=ENV, check=True)

# --- Trigger DINOv2 download to Drive cache (first run only, ~340 MB) ---
print('\nFetching DINOv2 weights to Drive cache (first run downloads ~340 MB)...')
_ = torch.hub.load(
    'facebookresearch/dinov2', 'dinov2_vitb14',
    pretrained=True, trust_repo=True,
)
print('✅ DINOv2 cached at', TORCH_HUB_CACHE)
del _

print(f'\n✅ v5 setup complete. Outputs will land under {V5_BASE}/')

In [ ]:
# === v5.2 — Train all four v5 runs (3 single-task + 1 multi-task) ===
# Robust: each run resumes from last.pt if it exists, and skips entirely
# if a best.pt is already on Drive.
import os, time

def _train_one(fam: str):
    out_dir = f'{V5_BASE}/{fam}'
    best_pt = f'{out_dir}/checkpoints/best.pt'
    last_pt = f'{out_dir}/checkpoints/last.pt'
    if os.path.isfile(best_pt) and not os.path.isfile(last_pt):
        print(f'\n=== {fam}: best.pt already present, skipping ===', flush=True)
        return

    print(f'\n=== Training v5/{fam} ===', flush=True)
    t0 = time.time()
    cmd = [
        'python', '-u', 'train.py',
        '--config', f'configs/track3_v5_{fam}.yaml',
        '--output-dir', out_dir,
        '--dataset-root', DATASET_ROOT,
        '--json-path', JSON_PATH,
        '--num-workers', '2',
    ]
    if os.path.isfile(last_pt):
        print(f'   (resuming from {last_pt})', flush=True)
        cmd += ['--resume', last_pt]
    run(cmd, label=f'train v5/{fam}')
    print(f'=== {fam} done in {(time.time() - t0)/60:.1f} min ===', flush=True)

for fam in V5_FAMILIES:
    _train_one(fam)

print('\n✅ All v5 trainings complete.')

In [ ]:
# === v5.3 — Evaluate all four v5 runs (val → tune thresholds → test flat & tuned → qualitative) ===
import os, json

def _eval_one(fam: str):
    out_dir = f'{V5_BASE}/{fam}'
    ckpt    = f'{out_dir}/checkpoints/best.pt'
    if not os.path.isfile(ckpt):
        print(f'\n(skip {fam}: no best.pt at {ckpt})', flush=True)
        return

    print(f'\n=== Phase 2 eval pipeline: v5/{fam} ===', flush=True)
    common = ['--config', f'configs/track3_v5_{fam}.yaml',
              '--ckpt', ckpt,
              '--output-dir', out_dir,
              '--dataset-root', DATASET_ROOT,
              '--json-path', JSON_PATH]

    # 1) eval val (flat 0.5) — dumps predictions_val.pt + eval_val.json + per_class_*_val.csv
    run(['python', '-u', 'eval.py'] + common + ['--split', 'val'],
        label=f'1/6 v5/{fam} eval val (flat 0.5)')

    # 2) tune per-class thresholds on val
    pred_val = f'{out_dir}/metrics/predictions_val.pt'
    thr_json = f'{out_dir}/metrics/thresholds_val.json'
    run(['python', '-u', '-m', 'src.scripts.tune_thresholds',
         '--predictions', pred_val, '--output', thr_json],
        label=f'2/6 v5/{fam} per-class threshold sweep on val')

    # 3) eval on test, flat 0.5 (primary number; comparable to v1/v4 flat 0.5)
    run(['python', '-u', 'eval.py'] + common + ['--split', 'test'],
        label=f'3/6 v5/{fam} eval test (flat 0.5) — primary')

    # 4) eval on test with val-tuned per-class thresholds (secondary number)
    run(['python', '-u', 'eval.py'] + common + ['--split', 'test', '--thresholds', thr_json],
        label=f'4/6 v5/{fam} eval test (val-tuned thresholds) — secondary')

    # 5) qualitative figures at flat 0.5 (matches v1 qualitative/ folder)
    run(['python', '-u', 'visualize.py'] + common + ['--split', 'test', '--num-samples', '20'],
        label=f'5/6 v5/{fam} visualize 20 samples (flat 0.5)')

    # 6) qualitative figures using val-tuned thresholds
    run(['python', '-u', 'visualize.py'] + common + ['--split', 'test', '--num-samples', '20',
                                                       '--thresholds', thr_json],
        label=f'6/6 v5/{fam} visualize 20 samples (val-tuned)')

for fam in V5_FAMILIES:
    _eval_one(fam)

# Compact summary table for the report
print('\n\n=== v5 vs v1 vs v3 vs v4 (test-set avg macro F1, flat 0.5 unless noted) ===')
print(f'{"variant":40s}  {"object":>8s}  {"event":>8s}  {"attr":>8s}  {"AVG":>8s}')

def _try_load(p):
    try:
        return json.load(open(p))
    except Exception:
        return None

# Reference numbers
print(f'{"v1 multitask (flat 0.5)":40s}    0.2850    0.2861    0.2404    0.2705')
print(f'{"v3 multitask flat 0.5":40s}    0.2246    0.2589    0.2238    0.2358')
print(f'{"v4 multitask (cleanup + v1 recipe)":40s}    0.2793    0.2801    0.2439    0.2677')

# v5 single-task — read the per-family scalar (its own family only)
print('  --- v5 single-task ---')
for fam in ['object', 'event', 'attribute']:
    s = _try_load(f'{V5_BASE}/{fam}/metrics/eval_test.json')
    if s is None:
        print(f'{"v5 " + fam + " single-task":40s}    MISSING')
        continue
    f = s.get(fam, {}).get('macro_f1', float('nan'))
    label = f'v5 {fam} single-task (flat 0.5)'
    print(f'{label:40s}    {f:8.4f}')

# v5 multi-task — all three families
print('  --- v5 multi-task ---')
mt_flat = _try_load(f'{V5_BASE}/multitask/metrics/eval_test.json')
mt_tun  = _try_load(f'{V5_BASE}/multitask/metrics/eval_test_tuned.json')
if mt_flat:
    o, e, a = (mt_flat[k]['macro_f1'] for k in ('object', 'event', 'attribute'))
    print(f'{"v5 multitask (flat 0.5)":40s}    {o:8.4f}  {e:8.4f}  {a:8.4f}  {(o+e+a)/3:8.4f}')
if mt_tun:
    o, e, a = (mt_tun[k]['macro_f1'] for k in ('object', 'event', 'attribute'))
    print(f'{"v5 multitask (val-tuned)":40s}    {o:8.4f}  {e:8.4f}  {a:8.4f}  {(o+e+a)/3:8.4f}')

# Track 4 — v6 (ConvNeXt-Tiny + Mamba SSM bi-temporal fusion + RAL loss)

**This section is fully self-contained.** On a fresh runtime, run the cells from here **in order** without touching anything above. No need to run the Track 1 / Track 2 / Track 3 cells.

**Hypothesis under test.** v1's concat-difference fusion and BCE loss are the bottlenecks. Mamba's selective state-space recurrence (causal, linear-time) treats the bi-temporal pair as a *sequence transition* — fundamentally different inductive bias from attention (v5) or concat-diff (v1). Robust Asymmetric Loss (Park 2023) is a different loss family than v3's failed LDAM-DRW.

**Architecture.** ConvNeXt-Tiny Siamese (UNFROZEN — v5 lesson: foundation features don't help on aerial bi-temporal; keep backbone trainable) → tokenize and interleave `[A_0, B_0, A_1, B_1, ...]` → 6 Mamba SSM blocks → mean-pool → linear family heads + changeflag + RAL loss.

(Note on backbone choice: docs/track4.md proposed full VMamba-Tiny, but VMamba requires custom CUDA kernels that are install-fragile on Colab. The Copilot-recommended variant — ConvNeXt encoder + Mamba *fusion* — captures the same SSM hypothesis with a far more reliable install. The `mamba-ssm` package + the official `Mamba` block do the heavy lifting.)

**Inductive-bias contrast across the project:**
- v1 = local CNN + concat-diff fusion + BCE
- v5 = ViT-attention everywhere + cross-attention fusion + ML-Decoder + BCE
- **v6 = local CNN + Mamba SSM fusion + linear heads + RAL** ← this section

**Outputs.** `/content/drive/MyDrive/dl_project_outputs/results/track4_v6/{object,event,attribute,multitask}/`

**Robustness.** Each cell auto-resumes from `last.pt` on Drive and skips families whose `best.pt` already exists, so re-running the section after a runtime disconnect is safe.

**Prerequisites.** Colab A100 runtime selected (*Runtime → Change runtime type → A100*). No other prerequisites — Track 4 v6 does not depend on any Track 1 / 2 / 3 artifacts.

## v6.1 — Setup (mount Drive, clone repo, install deps, build Mamba kernels)

Single cell that does everything. Idempotent — safe to re-run if you reconnect.

Mamba install (`causal-conv1d` + `mamba-ssm`) compiles CUDA kernels with `--no-build-isolation` so torch is visible during the build. If for any reason the compile still fails, the model auto-falls-back to a depthwise-conv mixer and training proceeds anyway. The cell prints whether it ended up with the real selective-scan kernels or the fallback.

In [ ]:
# === v6.1 — Setup (fully self-contained, no prior cells required) ===
# Idempotent: safe to re-run on the same runtime, and safe to run as the
# first cell on a fresh runtime without touching anything above.
import os, subprocess, sys

# --- Mount Drive ---
from google.colab import drive
drive.mount('/content/drive')

# --- Confirm GPU ---
print('\n--- nvidia-smi ---')
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'])
import torch
print(f'\ntorch.cuda.is_available(): {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# --- Paths ---
REPO_URL     = 'https://github.com/amirkiarafiei/deep-learning-project.git'
REPO_DIR     = '/content/deep-learning-project'
DRIVE_OUT    = '/content/drive/MyDrive/dl_project_outputs'
HF_CACHE     = f'{DRIVE_OUT}/hf_cache'
DATASET_ROOT = '/content/drive/MyDrive/dataset'
JSON_PATH    = f'{DATASET_ROOT}/dataset.json'

os.makedirs(HF_CACHE, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE

# --- Sanity-check critical paths ---
assert os.path.isfile(JSON_PATH), f'dataset.json not found at {JSON_PATH}'
print(f'\nDataset OK at {JSON_PATH}')

# --- Clone or update repo ---
if os.path.isdir(f'{REPO_DIR}/.git'):
    subprocess.run(['git', 'fetch', 'origin'], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'reset', '--hard', 'origin/main'], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--depth=1', REPO_URL, REPO_DIR], check=True)
print('\nCurrent commit:', end=' ')
subprocess.run(['git', 'log', '-1', '--oneline'], cwd=REPO_DIR)

# --- Output directories on Drive ---
V6_BASE     = f'{DRIVE_OUT}/results/track4_v6'
V6_FAMILIES = ['object', 'event', 'attribute', 'multitask']
for fam in V6_FAMILIES:
    for sub in ('checkpoints', 'logs', 'metrics', 'plots', 'qualitative'):
        os.makedirs(f'{V6_BASE}/{fam}/{sub}', exist_ok=True)

# --- Install Track-1 base deps (always required) ---
print('\n--- pip install (Track 1 base deps) ---')
base_deps = [
    'timm==1.0.27', 'scikit-learn>=1.4,<2.0', 'PyYAML>=6.0,<7.0', 'tqdm>=4.66',
    # Build deps needed for mamba-ssm's compile step:
    'ninja', 'packaging', 'wheel', 'setuptools',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + base_deps, check=True)
print('✅ base deps installed.')

# --- Install Mamba (best-effort, with fallback) ---
# mamba-ssm and causal-conv1d need torch at *build* time. The default pip
# install runs the build in an isolated env without torch → fails. The
# --no-build-isolation flag fixes this.
#
# If install still fails (rare hardware/CUDA version mismatch), our
# src/models/mamba_fusion.py automatically falls back to a depthwise-conv
# mixer. Training works either way; selective scan is faster.

def _install_mamba() -> bool:
    print('\n--- pip install (Mamba SSM) ---')
    print('  (this can take a couple of minutes — building CUDA kernels)')
    try:
        # Install causal-conv1d first (mamba-ssm imports it).
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '--no-build-isolation',
             'causal-conv1d>=1.4.0'],
            check=True, timeout=1500,
        )
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '--no-build-isolation',
             'mamba-ssm>=2.0.0'],
            check=True, timeout=1500,
        )
        # Verify the import works (catches "compiled but won't import" cases).
        from mamba_ssm import Mamba  # noqa: F401
        return True
    except Exception as e:
        print(f'\n⚠️  Mamba install/import failed: {type(e).__name__}: {e}')
        return False

mamba_ok = _install_mamba()
if mamba_ok:
    print('\n✅ mamba-ssm installed — v6 will use selective-scan CUDA kernels.')
else:
    print('\n⚠️  v6 will use the depthwise-conv fallback mixer.')
    print('   Training still works; ~30-40% slower forward pass per Mamba block,')
    print('   but the architecture (interleaved bi-temporal + stacked SSM-style')
    print('   mixers) is preserved. Numbers will be comparable.')

import timm, sklearn
print(f'\ntorch    {torch.__version__}  cuda={torch.cuda.is_available()}')
print(f'timm     {timm.__version__}')
print(f'sklearn  {sklearn.__version__}')

# --- Subprocess env preset reused by v6.2 and v6.3 ---
ENV = os.environ.copy()
ENV['HF_HOME'] = HF_CACHE
ENV['PYTHONUNBUFFERED'] = '1'
# Reduce memory fragmentation; useful for the Mamba SSM blocks which
# allocate / free intermediate state tensors heavily.
ENV['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

def run(cmd, label=None):
    if label: print(f'\n→ {label}', flush=True)
    subprocess.run(cmd, cwd=REPO_DIR, env=ENV, check=True)

print(f'\n✅ v6 setup complete. Outputs will land under {V6_BASE}/')

## v6.2 — Train all four v6 runs (3 single-task + 1 multi-task)

Iterates `[object, event, attribute, multitask]`. Each run:
- Auto-resumes from `last.pt` if Drive has one
- Skips entirely if `best.pt` is already on Drive (safe to re-run after a runtime disconnect)
- Writes checkpoint + per-epoch logs to Drive

Estimated A100 cost: ~50–60 min per single-task, ~80–90 min for multi-task. ~4.5 h end-to-end.

In [ ]:
# === v6.2 — Train all four v6 runs (3 single-task + 1 multi-task) ===
# Robust:
#  - Each run resumes from last.pt if it exists
#  - Skips entirely if best.pt is already on Drive
#  - Captures stdout+stderr so a crash prints the actual Python error
#  - FORCE_RESTART wipes Drive checkpoints once if you have broken weights
import os, shutil, time, subprocess

FORCE_RESTART = False   # ← Set to True once if you have broken v6 checkpoints to clear.

if FORCE_RESTART:
    for fam in V6_FAMILIES:
        ck_dir = f'{V6_BASE}/{fam}/checkpoints'
        if os.path.isdir(ck_dir):
            print(f'  Removing stale checkpoints: {ck_dir}', flush=True)
            shutil.rmtree(ck_dir)
            os.makedirs(ck_dir, exist_ok=True)

def _train_one(fam: str):
    out_dir = f'{V6_BASE}/{fam}'
    best_pt = f'{out_dir}/checkpoints/best.pt'
    last_pt = f'{out_dir}/checkpoints/last.pt'
    if os.path.isfile(best_pt) and not os.path.isfile(last_pt):
        print(f'\n=== {fam}: best.pt already present, skipping ===', flush=True)
        return

    print(f'\n=== Training v6/{fam} ===', flush=True)
    t0 = time.time()
    cmd = [
        'python', '-u', 'train.py',
        '--config', f'configs/track4_v6_{fam}.yaml',
        '--output-dir', out_dir,
        '--dataset-root', DATASET_ROOT,
        '--json-path', JSON_PATH,
        '--num-workers', '2',
    ]
    if os.path.isfile(last_pt):
        print(f'   (resuming from {last_pt})', flush=True)
        cmd += ['--resume', last_pt]

    # Stream stdout live but ALSO buffer it so we can print on failure.
    # This way the trainer's per-epoch progress is visible in real time,
    # and if it crashes we get the full traceback in the cell.
    print(f'→ train v6/{fam}', flush=True)
    proc = subprocess.Popen(
        cmd, cwd=REPO_DIR, env=ENV,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    buf = []
    for line in proc.stdout:
        print(line, end='', flush=True)
        buf.append(line)
        # Keep buffer bounded so we don't OOM on multi-hour runs.
        if len(buf) > 200:
            buf = buf[-200:]
    proc.wait()
    if proc.returncode != 0:
        # The last 200 lines are already printed above; this just calls
        # attention to the failure.
        print(f'\n❌ train v6/{fam} FAILED with exit code {proc.returncode}.')
        print(f'   See the last ~200 lines of training output above for the actual error.')
        raise SystemExit(f'v6/{fam} training failed; not continuing to other families.')

    print(f'=== {fam} done in {(time.time() - t0)/60:.1f} min ===', flush=True)

for fam in V6_FAMILIES:
    _train_one(fam)

print('\n✅ All v6 trainings complete.')

## v6.3 — Evaluate all four v6 runs

For each family: val flat → tune per-class thresholds on val → test flat (primary) → test val-tuned (secondary) → 2× qualitative figures. Prints the full comparison table v1 / v3 / v4 / v5 / v6 at the end.

Skips families whose `best.pt` is missing on Drive (you can run v6.3 standalone after a partial v6.2 to evaluate just what trained).

In [ ]:
# === v6.3 — Evaluate all four v6 runs (val → tune thresholds → test flat & tuned → qualitative) ===
import os, json

def _eval_one(fam: str):
    out_dir = f'{V6_BASE}/{fam}'
    ckpt    = f'{out_dir}/checkpoints/best.pt'
    if not os.path.isfile(ckpt):
        print(f'\n(skip {fam}: no best.pt at {ckpt})', flush=True)
        return

    print(f'\n=== Phase 2 eval pipeline: v6/{fam} ===', flush=True)
    common = ['--config', f'configs/track4_v6_{fam}.yaml',
              '--ckpt', ckpt,
              '--output-dir', out_dir,
              '--dataset-root', DATASET_ROOT,
              '--json-path', JSON_PATH]

    run(['python', '-u', 'eval.py'] + common + ['--split', 'val'],
        label=f'1/6 v6/{fam} eval val (flat 0.5)')

    pred_val = f'{out_dir}/metrics/predictions_val.pt'
    thr_json = f'{out_dir}/metrics/thresholds_val.json'
    run(['python', '-u', '-m', 'src.scripts.tune_thresholds',
         '--predictions', pred_val, '--output', thr_json],
        label=f'2/6 v6/{fam} per-class threshold sweep on val')

    run(['python', '-u', 'eval.py'] + common + ['--split', 'test'],
        label=f'3/6 v6/{fam} eval test (flat 0.5) — primary')

    run(['python', '-u', 'eval.py'] + common + ['--split', 'test', '--thresholds', thr_json],
        label=f'4/6 v6/{fam} eval test (val-tuned thresholds) — secondary')

    run(['python', '-u', 'visualize.py'] + common + ['--split', 'test', '--num-samples', '20'],
        label=f'5/6 v6/{fam} visualize 20 samples (flat 0.5)')

    run(['python', '-u', 'visualize.py'] + common + ['--split', 'test', '--num-samples', '20',
                                                       '--thresholds', thr_json],
        label=f'6/6 v6/{fam} visualize 20 samples (val-tuned)')

for fam in V6_FAMILIES:
    _eval_one(fam)

# Compact summary table — all four versions for the report
print('\n\n=== v6 vs v1 vs v3 vs v4 vs v5 (test-set avg macro F1, flat 0.5 unless noted) ===')
print(f'{"variant":40s}  {"object":>8s}  {"event":>8s}  {"attr":>8s}  {"AVG":>8s}')

def _try_load(p):
    try:
        return json.load(open(p))
    except Exception:
        return None

print(f'{"v1 multitask (flat 0.5)":40s}    0.2850    0.2861    0.2404    0.2705')
print(f'{"v3 multitask flat 0.5":40s}    0.2246    0.2589    0.2238    0.2358')
print(f'{"v4 multitask (cleanup + v1 recipe)":40s}    0.2793    0.2801    0.2439    0.2677')
print(f'{"v5 multitask (DINOv2 + cross-attn)":40s}    0.2312    0.2509    0.2121    0.2314')

print('  --- v6 single-task ---')
for fam in ['object', 'event', 'attribute']:
    s = _try_load(f'{V6_BASE}/{fam}/metrics/eval_test.json')
    if s is None:
        print(f'{"v6 " + fam + " single-task":40s}    MISSING')
        continue
    f = s.get(fam, {}).get('macro_f1', float('nan'))
    print(f'{"v6 " + fam + " single-task (flat 0.5)":40s}    {f:8.4f}')

print('  --- v6 multi-task ---')
mt_flat = _try_load(f'{V6_BASE}/multitask/metrics/eval_test.json')
mt_tun  = _try_load(f'{V6_BASE}/multitask/metrics/eval_test_tuned.json')
if mt_flat:
    o, e, a = (mt_flat[k]['macro_f1'] for k in ('object', 'event', 'attribute'))
    print(f'{"v6 multitask (flat 0.5)":40s}    {o:8.4f}  {e:8.4f}  {a:8.4f}  {(o+e+a)/3:8.4f}')
if mt_tun:
    o, e, a = (mt_tun[k]['macro_f1'] for k in ('object', 'event', 'attribute'))
    print(f'{"v6 multitask (val-tuned)":40s}    {o:8.4f}  {e:8.4f}  {a:8.4f}  {(o+e+a)/3:8.4f}')